In [4]:
import greengraph as gg
import numpy as np
import xarray as xr

In [5]:
from greengraph.importers.databases.inputoutput import useeio
from greengraph.importers.databases.generic import graph_system_from_input_output_matrices

dct = useeio.load_useeio_data_from_zenodo(version='2.0.1-411')
G = graph_system_from_input_output_matrices(
    name_system='useeio',
    assign_new_uuids=True,
    str_extension_nodes_uuid='name',
    str_production_nodes_uuid='name',
    str_indicator_nodes_uuid='name',
    matrix_convention='I-A',
    array_production=dct['A'].to_numpy(),
    array_extension=dct['B'].to_numpy(),
    array_indicator=dct['C'].to_numpy(),
    list_dicts_production_node_metadata=dct['dicts_A_metadata'],
    list_dicts_extension_node_metadata=dct['dicts_B_metadata'],
    list_dicts_indicator_node_metadata=dct['dicts_C_metadata'],
)

greengraph | INFO | Found file USEEIOv2.0.1-411.xlsx in local cache.
greengraph | INFO | 13:12:55: Started extracting USEEIO data from Excel file.
greengraph | INFO | 13:13:02: Completed extracting USEEIO data from Excel file. (00:06 min:sec)
greengraph | INFO | 13:13:02: Started modifying USEEIO data.
greengraph | INFO | 13:13:02: Completed modifying USEEIO data. (00:00 min:sec)
greengraph | INFO | 13:13:02: Started creating MultiDiGraph from technosphere matrix.
greengraph | INFO | # of nodes: 411, # of edges: 92,527
greengraph | INFO | 13:13:02: Started creating graph from adjacency matrix (same row/column labels).
greengraph | INFO | 13:13:02: Completed creating graph from adjacency matrix (same row/column labels). (00:00 min:sec)
greengraph | INFO | 13:13:02: Completed creating MultiDiGraph from technosphere matrix. (00:00 min:sec)
greengraph | INFO | 13:13:02: Started creating MultiDiGraph from biosphere matrix.
greengraph | INFO | # of nodes: 2722, # of edges: 204,507
greengraph

In [6]:
type(G)

greengraph.core.GreenMultiDiGraph

In [7]:
[(n, attr) for n, attr in G.nodes(data=True) if attr['type'] == 'production'][0]

('25210612-cd13-4979-b9f9-a7881ce39e38',
 {'name': 'Fresh soybeans, canola, flaxseeds, and other oilseeds',
  'location': 'US',
  'code': '1111A0',
  'category': '11: Agriculture, Forestry, Fishing and Hunting/1111: Oilseed and Grain Farming',
  'unit': 'USD',
  'annual production': 42656016000.0,
  'uuid': '25210612-cd13-4979-b9f9-a7881ce39e38',
  'index': 0,
  'type': 'production',
  'system': 'useeio',
  'production': 1.0})

In [15]:
M = G.generate_matrix_system(
    matrixformat='dense',
    generate_A=True,
    generate_B=True,
    generate_Q=True,
    A_sort_attributes=['index'],
    B_sort_attributes=['index'],
    Q_sort_attributes=['index'],
)

greengraph | INFO | 11:13:51: Started Generating production matrix.
greengraph | INFO | 11:13:51: Completed Generating production matrix. (00:00 min:sec)
greengraph | INFO | 11:13:51: Started Normalizing production matrix ('I-A'-convention).
greengraph | INFO | 11:13:51: Completed Normalizing production matrix ('I-A'-convention). (00:00 min:sec)
greengraph | INFO | 11:13:51: Started Generating biosphere matrix.
greengraph | INFO | 11:13:51: Completed Generating biosphere matrix. (00:00 min:sec)
greengraph | INFO | 11:13:51: Started Normalizing biosphere matrix ('I-B'-convention).
greengraph | INFO | 11:13:51: Completed Normalizing biosphere matrix ('I-B'-convention). (00:00 min:sec)
greengraph | INFO | 11:13:51: Started Generating characterization matrix.
greengraph | INFO | 11:13:51: Completed Generating characterization matrix. (00:00 min:sec)


In [18]:
M.matrices.keys()

dict_keys(['A', 'Anorm', 'B', 'Bnorm', 'Q'])

In [16]:
type(M)

greengraph.core.GreenMatrixContainer

In [23]:
G.get_node_by_attributes(
    type='production',
)

AttributeError: Multiple nodes found matching given attributes. Please refine attributes.

In [19]:
G.get_random_node(type='production', data=True)

('74fad07f-8ee6-4bf1-9152-f74bb1534b39',
 {'name': 'Employment services',
  'location': 'US',
  'code': '561300',
  'category': '56: Administrative and Support and Waste Management and Remediation Services/5613: Employment Services',
  'unit': 'USD',
  'annual production': 247715014500.0,
  'uuid': '74fad07f-8ee6-4bf1-9152-f74bb1534b39',
  'index': 344,
  'type': 'production',
  'system': 'useeio',
  'production': 1.0})

In [24]:
M.lca(demand={G.get_random_node(type='production', data=False): 1.0})
M.lcia()

greengraph | INFO | 11:17:00: Started calculating production vector.
greengraph | INFO | 11:17:00: Completed calculating production vector. (00:00 min:sec)
greengraph | INFO | 11:17:00: Started calculating inventory vector.
greengraph | INFO | 11:17:00: Completed calculating inventory vector. (00:00 min:sec)
greengraph | INFO | 11:17:00: Started calculating impact vector.
greengraph | INFO | 11:17:00: Completed calculating impact vector. (00:00 min:sec)


In [25]:
M.matrices['h']

<xarray.DataArray (indicator nodes: 23)> Size: 184B
array([2.11506491e-04, 3.24932079e-03, 1.79781862e-02, 8.15914635e-04,
       2.09801073e+00, 7.61775331e-05, 1.90994748e-02, 5.49135397e+00,
       1.20209776e-01, 1.34440290e-05, 3.53880724e-11, 9.69713742e-10,
       3.80951190e-05, 1.00510181e-09, 4.74444608e-06, 8.80440742e-02,
       2.62839615e-02, 1.78636486e+00, 1.67582263e-08, 5.90025893e-07,
       3.11645870e-01, 3.28995359e-03, 9.87076608e-01])
Coordinates:
  * indicator nodes  (indicator nodes) <U36 3kB 'de2455e1-e3ef-46e3-b400-8f3b...

In [10]:
G.get_node_by_attributes(name='Acidification Potential')

'74e69688-dca9-4414-91bd-7f1b22989ec8'

In [26]:
len(G.edges())

302307

In [30]:
M.matrices['h'].coords['indicator nodes'].values

array(['de2455e1-e3ef-46e3-b400-8f3b0033f659',
       '5575b356-b63f-493c-8182-b5d3b90a8e07',
       'aa9c731d-a9ca-4da4-b7e3-35891738573f',
       '0fd9caef-6501-4e21-accc-7f45adbbd171',
       '14ebb0b9-7976-49e7-ac99-acf6185a2dd2',
       'e94e9f9b-1e81-4e68-8cda-58e4b015c9a8',
       'd10b79b9-684f-4af7-9a9a-3587fabd32e2',
       'dc0f16d0-b023-4726-994a-f556e7d58de0',
       '4b4ed003-c09b-4f6b-ae9a-f463654fdb0c',
       '972104d8-f8b2-4af4-8916-9de623f8fc52',
       '407193c3-2ba6-43c9-840b-b44fdcce7d35',
       '2dd2e9eb-c8b2-423b-88dc-acd835cac839',
       '394dc34f-21c5-4adb-bd3f-fb81e67f4c54',
       '539cc16b-90c7-4d22-957c-1bbc0cde8d28',
       '20d0bae1-8e11-43f6-8e5c-bae9b042f32f',
       'fda9e924-1652-4dcf-a511-1ef454fdbe2d',
       '3e791f5b-e75a-41e7-b308-0e22ffce2494',
       '10bdf16f-67cc-4f3a-a705-10199ed9ae6c',
       'e0bcfdf6-9ef3-4491-a0fa-a7aff6dc4775',
       'e7c7a510-645e-4e20-a709-2fe7a7553294',
       'b238cdca-c553-4472-bf70-5ec3a63e6244',
       '602e5

In [54]:
M.matrices['h'].sel({'indicator nodes': 'de2455e1-e3ef-46e3-b400-8f3b0033f659'}).item()

0.00021150649057300334

In [34]:
node_imp = G.get_node_by_attributes(name='Acidification Potential')

In [36]:
M.matrices['h'].sel({'indicator nodes': node_imp}).values

array(0.00021151)

In [41]:
type(G.nodes[M.matrices['h'].coords['indicator nodes'].values[0]])

dict

In [57]:
import pandas as pd

In [58]:
pd.DataFrame(
    [
        {**G.nodes[node], 'amount': M.matrices['h'].sel({'indicator nodes': node}).item()}
        for node in M.matrices['h'].coords['indicator nodes'].values
    ]
)

,name,code,unit,group,description,uuid,index,type,system,amount
0,Acidification Potential,ACID,kg SO2 eq,Impact Potential,Acid Rain,de2455e1-e3ef-46e3-b400-8f3b0033f659,0,indicator,useeio,2.115065e-04
1,Commercial Construction and Demolition Debris,CCDD,kg,Waste Generated,Construction Debris,5575b356-b63f-493c-8182-b5d3b90a8e07,1,indicator,useeio,3.249321e-03
2,Commercial Municipal Solid Waste,CMSW,kg,Waste Generated,Municipal Solid Waste,aa9c731d-a9ca-4da4-b7e3-35891738573f,2,indicator,useeio,1.797819e-02
3,Commercial RCRA Hazardous Waste,CRHW,kg,Waste Generated,Hazardous Waste,0fd9caef-6501-4e21-accc-7f45adbbd171,3,indicator,useeio,8.159146e-04
4,Energy Use,ENRG,MJ,Resource Use,Energy Use,14ebb0b9-7976-49e7-ac99-acf6185a2dd2,4,indicator,useeio,2.098011e+00
5,Eutrophication Potential,EUTR,kg N eq,Impact Potential,Water Eutrophication,e94e9f9b-1e81-4e68-8cda-58e4b015c9a8,5,indicator,useeio,7.617753e-05
6,Freshwater Ecotoxicity Potential,ETOX,CTUe,Impact Potential,Freshwater Ecotoxicity,d10b79b9-684f-4af7-9a9a-3587fabd32e2,6,indicator,useeio,1.909947e-02
7,Freshwater withdrawals,WATR,kg,Resource Use,Water Use,dc0f16d0-b023-4726-994a-f556e7d58de0,7,indicator,useeio,5.491354e+00
8,Greenhouse Gases,GHG,kg CO2 eq,Impact Potential,Greenhouse Gases,4b4ed003-c09b-4f6b-ae9a-f463654fdb0c,8,indicator,useeio,1.202098e-01
9,Hazardous Air Pollutants,HAPS,kg,Chemical Releases,Hazardous Air Pollutants,972104d8-f8b2-4af4-8916-9de623f8fc52,9,indicator,useeio,1.344403e-05


In [3]:
M['h']

NameError: name 'M' is not defined